---
title: Глава 6. Работа с множествами
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-08-17
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
---

In [2]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="********",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False

%sql engine

print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


:::{note} Примечание
Термины _множество_ и _набор_ в данной книге являются синонимами.
:::

## Основы теории множеств

- A `union` B – объединение
- A `intersect` B – пересечение; удаляет все повторяющиеся строки, обнаруженные в области перекрытия наборов данных.
- А `except` B – исключение; возвращает первый результирующий набор за вычетом любого перекрытия со вторым результирующим набором.

## Теория множеств на практике

In [3]:
%%sql
desc customer;

9 rows affected.

Field,Type,Null,Key,Default,Extra
customer_id,smallint unsigned,NO,PRI,None,auto_increment
store_id,tinyint unsigned,NO,MUL,None,
first_name,varchar(45),NO,,None,
last_name,varchar(45),NO,MUL,None,
email,varchar(50),YES,,None,
address_id,smallint unsigned,NO,MUL,None,
active,tinyint(1),NO,,1,
create_date,datetime,NO,,None,
last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [4]:
%%sql
desc city;

4 rows affected.

Field,Type,Null,Key,Default,Extra
city_id,smallint unsigned,NO,PRI,None,auto_increment
city,varchar(50),NO,,None,
country_id,smallint unsigned,NO,MUL,None,
last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


:::{important} При выполнении операции над двумя наборами данных
- Оба набора данных должны иметь одинаковое количество столбцов.
- Типы данных каждого столбца в двух наборах данных должны быть одинаковыми _(либо сервер должен иметь возможность преобразовывать один тип в другой)_.
:::

In [6]:
%%sql
SELECT 1 num, 'abc' str
UNION
SELECT 9 num, 'xyz' str;

2 rows affected.

num,str
1,abc
9,xyz


## Операторы для работы с множествами

### Оператор *union*

- `union` – сортирует объединенный набор и удаляет дубликаты
- `union all` – этого не делает, оставляет перекрывающиеся данные

In [7]:
import pandas as pd
print(pd.__version__)

3.0.5


In [9]:
%config SqlMagic.autopandas = True

In [10]:
%%sql
SELECT 'CUST' typ, c.first_name, c.last_name
FROM customer c
UNION ALL
SELECT 'ACTR' typ, a.first_name, a.last_name
FROM actor a;

799 rows affected.

,typ,first_name,last_name
0,CUST,MARY,SMITH
1,CUST,PATRICIA,JOHNSON
2,CUST,LINDA,WILLIAMS
3,CUST,BARBARA,JONES
4,CUST,ELIZABETH,BROWN
...,...,...,...
794,ACTR,BELA,WALKEN
795,ACTR,REESE,WEST
796,ACTR,MARY,KEITEL
797,ACTR,JULIA,FAWCETT


Запрос возвращает 799 строк: 599 строк из таблицы _customer_ и 200 строк из таблицы _actor_.

In [11]:
%%sql
SELECT 'ACTR' typ, a.first_name, a.last_name
FROM actor a
UNION ALL
SELECT 'ACTR' typ, a.first_name, a.last_name
FROM actor a;

400 rows affected.

,typ,first_name,last_name
0,ACTR,PENELOPE,GUINESS
1,ACTR,NICK,WAHLBERG
2,ACTR,ED,CHASE
3,ACTR,JENNIFER,DAVIS
4,ACTR,JOHNNY,LOLLOBRIGIDA
...,...,...,...
395,ACTR,BELA,WALKEN
396,ACTR,REESE,WEST
397,ACTR,MARY,KEITEL
398,ACTR,JULIA,FAWCETT


200 строк из таблицы _actor_ включаются в результирующий набор _дважды_.

In [18]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
UNION ALL
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%';

5 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS
1,JENNIFER,DAVIS
2,JUDY,DEAN
3,JODIE,DEGENERES
4,JULIANNE,DENCH


Из пяти строк одна является дубликатом JENNIFER DAVIS. Чтобы исключить повторяющиеся строки воспользуемся UNION:

In [14]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
UNION
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%';

4 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS
1,JUDY,DEAN
2,JODIE,DEGENERES
3,JULIANNE,DENCH


### Оператор *intersect*

Удаляет все повторяющиеся строки, обнаруженные в области перекрытия наборов данных.

In [15]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
INTERSECT
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%';

1 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS


Пересечение двух запросов дает единственное имя JENNIFER DAVIS, имеющееся в результирующих наборах обоих запросов.

In [16]:
%%sql
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'D%' AND c.last_name LIKE 'T%'
INTERSECT
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'D%' AND a.last_name LIKE 'T%';

,first_name,last_name


### Оператор *except*

Возвращает первый результирующий набор за вычетом любого перекрытия со вторым результирующим набором.

In [20]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
EXCEPT
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%';

3 rows affected.

,first_name,last_name
0,JUDY,DEAN
1,JODIE,DEGENERES
2,JULIANNE,DENCH


## Правила применения операторов для работы с множествами

### Сортировка результатов составного запроса

При указании имен столбцов в предложении _order by_ нужно выбирать одно из имен столбцов в первом запросе составного запроса.

In [26]:
%%sql
SELECT a.first_name fname, a.last_name lname
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION ALL
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
ORDER BY lname, fname;

5 rows affected.

,fname,lname
0,JENNIFER,DAVIS
1,JENNIFER,DAVIS
2,JUDY,DEAN
3,JODIE,DEGENERES
4,JULIANNE,DENCH


Если в предложении _order by_ указать имя столбца из второго запроса, будет выведено сообщение об ошибке:

In [29]:
%%sql
SELECT a.first_name fname, a.last_name lname
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION ALL
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
ORDER BY last_name, first_name;

RuntimeError: (pymysql.err.OperationalError) (1054, "Unknown column 'last_name' in 'order clause'")
[SQL: SELECT a.first_name fname, a.last_name lname
FROM actor a
WHERE a.first_name LIKE 'J%%' AND a.last_name LIKE 'D%%'
UNION ALL
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%%' AND c.last_name LIKE 'D%%'
ORDER BY last_name, first_name;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


:::{tip} Рекомендация
Давать столбцам в обоих запросах одинаковые псевдонимы столбцов, чтобы избежать указанной проблемы.
:::

In [31]:
%%sql
SELECT a.first_name fname, a.last_name lname
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION ALL
SELECT c.first_name fname, c.last_name lname
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
ORDER BY lname, fname;

5 rows affected.

,fname,lname
0,JENNIFER,DAVIS
1,JENNIFER,DAVIS
2,JUDY,DEAN
3,JODIE,DEGENERES
4,JULIANNE,DENCH


:::{important} Алиасы
Если имена столбцов в обоих таблицах совпадают, то можно не давать псевдонимы.

Но если объединять таблицы с разными именами колонок, то без алиасов в первом запросе не обойтись. Поэтому первому запросу рекомендуется **всегда давать четкие алиасы** ради надёжности кода и защиты от потенциальных ошибок. 
:::

In [32]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION ALL
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
ORDER BY last_name, first_name;

5 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS
1,JENNIFER,DAVIS
2,JUDY,DEAN
3,JODIE,DEGENERES
4,JULIANNE,DENCH


### Приоритеры операций над множествами

:::{important}
Если составной запрос содержит более двух запросов с использованием разных операторов для работы с множествами, следует **подумать** о порядке размещения запросов в составном запросе для достижения желаемых результатов.
:::

In [33]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION ALL
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'M%' AND a.last_name LIKE 'T%'
UNION
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%';

6 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS
1,JUDY,DEAN
2,JODIE,DEGENERES
3,JULIANNE,DENCH
4,MARY,TANDY
5,MENA,TEMPLE


**Размещение операторов имеет значение**. \
Вот тот же составной запрос с обратным размещением операторов:

In [34]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'M%' AND a.last_name LIKE 'T%'
UNION ALL
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%';

7 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS
1,JUDY,DEAN
2,JODIE,DEGENERES
3,JULIANNE,DENCH
4,MARY,TANDY
5,MENA,TEMPLE
6,JENNIFER,DAVIS


:::{attention} Как правило
Составные запросы, содержащие три или более запросов, вычисляются в порядке сверху-вниз, но со следующими предостережениями:
- Спецификация ANSI SQL требует, чтобы оператор `intersect` имел приоритет над другими операторами множества.
- Порядок, в котором выполняется объединение запросов, можно указать с помощью скобок.
:::

In [35]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'J%' AND a.last_name LIKE 'D%'
UNION
(SELECT a.first_name, a.last_name
FROM actor a
WHERE a.first_name LIKE 'M%' AND a.last_name LIKE 'T%'
UNION ALL
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.first_name LIKE 'J%' AND c.last_name LIKE 'D%'
);

6 rows affected.

,first_name,last_name
0,JENNIFER,DAVIS
1,JUDY,DEAN
2,JODIE,DEGENERES
3,JULIANNE,DENCH
4,MARY,TANDY
5,MENA,TEMPLE



---

## Упражнения

### Упражнение 6.1

Пусть множество А = {L, M, N, O, P}, а множество В = {P, Q, R, S, T}. Какие множества будут сгенерированы следующими операциями?
- A union B
- A union all B
- A intersect B
- A except B

In [39]:
%config SqlMagic.autopandas = False

In [55]:
%%sql
SELECT column_0 AS my_set   -- 3. Внешний запрос: берет колонку и переименовывает её
FROM (                      -- 1. Создаем виртуальную таблицу "на лету"
    VALUES ROW('L'), ROW('M'), ROW('N'), ROW('O'), ROW('P')
    UNION
    VALUES ROW('P'), ROW('Q'), ROW('R'), ROW('S'), ROW('T')
) sub;                      -- 2. Называем всю эту виртуальную таблицу именем "sub"

9 rows affected.

my_set
L
M
N
O
P
Q
R
S
T


In [42]:
%%sql
SELECT column_0 AS my_set
FROM (
    VALUES ROW('L'), ROW('M'), ROW('N'), ROW('O'), ROW('P')
    UNION ALL
    VALUES ROW('P'), ROW('Q'), ROW('R'), ROW('S'), ROW('T')
) sub;

10 rows affected.

my_set
L
M
N
O
P
P
Q
R
S
T


In [43]:
%%sql
SELECT column_0 AS my_set
FROM (
    VALUES ROW('L'), ROW('M'), ROW('N'), ROW('O'), ROW('P')
    INTERSECT
    VALUES ROW('P'), ROW('Q'), ROW('R'), ROW('S'), ROW('T')
) sub;

1 rows affected.

my_set
P


In [44]:
%%sql
SELECT column_0 AS my_set
FROM (
    VALUES ROW('L'), ROW('M'), ROW('N'), ROW('O'), ROW('P')
    EXCEPT
    VALUES ROW('P'), ROW('Q'), ROW('R'), ROW('S'), ROW('T')
) sub;

4 rows affected.

my_set
L
M
N
O



---

### Упражнение 6.2

Напишите составной запрос, который находит имена и фамилии всех актеров и клиентов, чьи фамилии начинаются с буквы _**L**_.

In [57]:
%config SqlMagic.displaylimit = 25

In [52]:
%%sql
SELECT a.first_name, a.last_name
FROM actor a
WHERE a.last_name LIKE 'L%'
UNION
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.last_name LIKE 'L%';

24 rows affected.

first_name,last_name
MATTHEW,LEIGH
JOHNNY,LOLLOBRIGIDA
MISTY,LAMBERT
JACOB,LANCE
RENEE,LANE
HEIDI,LARSON
DARYL,LARUE
LAURIE,LAWRENCE
JEANNE,LAWSON
LAWRENCE,LAWTON



---

### Упражнение 6.3

Отсортируйте результаты выполнения упражнения 6.2 по столбцу _last_name_.

In [58]:
%%sql
SELECT a.first_name fname, a.last_name lname
FROM actor a
WHERE a.last_name LIKE 'L%'
UNION
SELECT c.first_name, c.last_name
FROM customer c
WHERE c.last_name LIKE 'L%'
ORDER BY lname, fname;

24 rows affected.

fname,lname
MISTY,LAMBERT
JACOB,LANCE
RENEE,LANE
HEIDI,LARSON
DARYL,LARUE
LAURIE,LAWRENCE
JEANNE,LAWSON
LAWRENCE,LAWTON
KIMBERLY,LEE
MATTHEW,LEIGH



---